# EfficientNetB0 — Brain Tumor MRI Classification

**Model Owner:** Hettiarachchi H.A.A.S  
**IT Number:** IT25104064  
**Assigned Model:** EfficientNetB0

This notebook contains **two separately trained variants** of the assigned model.  
The better variant is selected using **Validation Macro F1**, and the Official Test set is used only after variant selection.

## 1. Project and Model Overview

**Model:** EfficientNetB0  
**Weights:** ImageNet  
**Classes:** glioma, meningioma, no-tumor, pituitary

This is one of the two intentionally strong transfer-learning models in the final project.

## 2. Why EfficientNetB0 Is Suitable

EfficientNetB0 includes its required input rescaling inside the Keras application model.

The notebook uses:
- ImageNet transfer learning,
- AdamW,
- label smoothing,
- training-only augmentation,
- frozen-backbone warm-up,
- deeper low-learning-rate fine-tuning,
- Early Stopping,
- ReduceLROnPlateau,
- selection by Validation Macro F1.

These choices are intended to improve generalization rather than manually alter evaluation results.

## 3. Install Required Libraries

In [ ]:
%pip -q install kagglehub scikit-learn

## 4. Import Libraries and Set Reproducibility

In [ ]:
from pathlib import Path
import json
import shutil
import zipfile
import random
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import kagglehub

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

KAGGLE_DATASET = "maleeshaweerakkody/brain-tumor-mri-preprocessed"

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
# ------------------------------------------------------------
# Model ownership
# ------------------------------------------------------------
MODEL_OWNER = "Hettiarachchi H.A.A.S"
OWNER_IT_NUMBER = "IT25104064"
OWNER_MEMBER_NO = OWNER_MEMBER_NO

print("Model Owner:", MODEL_OWNER)
print("IT Number:", OWNER_IT_NUMBER)
print("Assigned Model:", "EfficientNetB0")


## 5. Download Final Preprocessed Kaggle Dataset

In [ ]:
KAGGLE_ROOT = Path(
    kagglehub.dataset_download(
        KAGGLE_DATASET
    )
)

WORK_DIR = Path(
    "/content/brain_tumor_high_performance"
)

DATA_ROOT = WORK_DIR / "dataset"
MODEL_DIR = WORK_DIR / "models"
RESULTS_DIR = WORK_DIR / "results"

for p in [
    DATA_ROOT,
    MODEL_DIR,
    RESULTS_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

zip_files = list(
    KAGGLE_ROOT.rglob("*.zip")
)

if zip_files:
    for zpath in zip_files:
        with zipfile.ZipFile(
            zpath,
            "r"
        ) as zf:
            zf.extractall(
                DATA_ROOT
            )

    SEARCH_ROOT = DATA_ROOT
else:
    SEARCH_ROOT = KAGGLE_ROOT

print("Dataset root:", SEARCH_ROOT)

## 6. Detect Training, Validation and Official Test Folders

In [ ]:
def find_split_dir(
    root,
    names
):
    names = {
        x.lower()
        for x in names
    }

    matches = [
        p
        for p in root.rglob("*")
        if p.is_dir()
        and p.name.lower() in names
    ]

    if not matches:
        return None

    return sorted(
        matches,
        key=lambda p: (
            -len([
                x
                for x in p.iterdir()
                if x.is_dir()
            ]),
            len(p.parts)
        )
    )[0]


TRAIN_DIR = find_split_dir(
    SEARCH_ROOT,
    ["train", "training"]
)

VAL_DIR = find_split_dir(
    SEARCH_ROOT,
    ["validation", "valid", "val"]
)

TEST_DIR = find_split_dir(
    SEARCH_ROOT,
    ["test", "testing"]
)

print("Training:", TRAIN_DIR)
print("Validation:", VAL_DIR)
print("Official Test:", TEST_DIR)

assert TRAIN_DIR is not None
assert VAL_DIR is not None
assert TEST_DIR is not None

## 7. EDA and Class Distribution

In [ ]:
IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff"
}

CLASS_NAMES = sorted([
    p.name
    for p in TRAIN_DIR.iterdir()
    if p.is_dir()
])

NUM_CLASSES = len(
    CLASS_NAMES
)

assert NUM_CLASSES == 4

print("Exact class order:")
for i, class_name in enumerate(
    CLASS_NAMES
):
    print(
        i,
        "->",
        class_name
    )


def count_split(
    root
):
    rows = []

    for class_name in CLASS_NAMES:
        count = sum(
            1
            for p in (
                root / class_name
            ).rglob("*")
            if p.is_file()
            and p.suffix.lower()
            in IMAGE_EXTS
        )

        rows.append({
            "Class": class_name,
            "Count": count
        })

    return pd.DataFrame(
        rows
    )


counts_df = (
    count_split(
        TRAIN_DIR
    ).rename(
        columns={
            "Count": "Training"
        }
    )
    .merge(
        count_split(
            VAL_DIR
        ).rename(
            columns={
                "Count": "Validation"
            }
        ),
        on="Class"
    )
    .merge(
        count_split(
            TEST_DIR
        ).rename(
            columns={
                "Count": "Official Test"
            }
        ),
        on="Class"
    )
)

display(
    counts_df
)

counts_df.set_index(
    "Class"
).plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title(
    "Brain Tumor MRI — Class Distribution"
)

plt.ylabel(
    "Images"
)

plt.xticks(
    rotation=20
)

plt.tight_layout()
plt.show()

## 8. Build TensorFlow Datasets and Verify Exact Class Order

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    image_size=(
        IMG_SIZE,
        IMG_SIZE
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    image_size=(
        IMG_SIZE,
        IMG_SIZE
    ),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    image_size=(
        IMG_SIZE,
        IMG_SIZE
    ),
    batch_size=BATCH_SIZE,
    shuffle=False
)

train_ds = train_ds.prefetch(
    AUTOTUNE
)

val_ds = val_ds.prefetch(
    AUTOTUNE
)

test_ds = test_ds.prefetch(
    AUTOTUNE
)

images, labels = next(
    iter(
        train_ds
    )
)

print(
    "Image shape:",
    images.shape
)

print(
    "Input pixel range:",
    float(
        tf.reduce_min(
            images
        )
    ),
    "to",
    float(
        tf.reduce_max(
            images
        )
    )
)

## 9. Model-Specific Preprocessing and Training-Only Augmentation

The final Kaggle dataset has already completed the group's common image preprocessing.

Inside this notebook:
- Validation/Test are not augmented.
- Training receives mild or moderate augmentation depending on the variant.
- EfficientNetB0 includes its required input rescaling inside the Keras application model.

The saved `.keras` model therefore keeps its architecture-specific preprocessing consistent with deployment.

In [ ]:
plt.figure(
    figsize=(9, 6)
)

for i in range(
    min(
        6,
        images.shape[
            0
        ]
    )
):
    plt.subplot(
        2,
        3,
        i + 1
    )

    plt.imshow(
        tf.cast(
            images[
                i
            ],
            tf.uint8
        )
    )

    class_index = int(
        tf.argmax(
            labels[
                i
            ]
        )
    )

    plt.title(
        CLASS_NAMES[
            class_index
        ]
    )

    plt.axis(
        "off"
    )

plt.tight_layout()
plt.show()

## 10. Evaluation Metrics and Helper Functions

In [ ]:
def macro_f1():
    return tf.keras.metrics.F1Score(
        average="macro",
        name="f1_macro"
    )


def make_callbacks(
    checkpoint_path,
    patience
):
    return [
        keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor="val_f1_macro",
            mode="max",
            save_best_only=True,
            verbose=1
        ),

        keras.callbacks.EarlyStopping(
            monitor="val_f1_macro",
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=1
        ),

        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
    ]


def compile_model(
    model,
    learning_rate,
    weight_decay
):
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay
        ),
        loss=keras.losses.CategoricalCrossentropy(
            label_smoothing=0.05
        ),
        metrics=[
            keras.metrics.CategoricalAccuracy(
                name="accuracy"
            ),
            macro_f1()
        ]
    )


def get_predictions(
    model,
    dataset
):
    y_true = []
    probabilities = []

    for x_batch, y_batch in dataset:
        p = model.predict(
            x_batch,
            verbose=0
        )

        y_true.extend(
            np.argmax(
                y_batch.numpy(),
                axis=1
            )
        )

        probabilities.extend(
            p
        )

    return (
        np.asarray(
            y_true,
            dtype=np.int64
        ),
        np.asarray(
            probabilities,
            dtype=np.float32
        )
    )


def calculate_metrics(
    y_true,
    probabilities
):
    y_pred = np.argmax(
        probabilities,
        axis=1
    )

    return {
        "Accuracy": float(
            accuracy_score(
                y_true,
                y_pred
            )
        ),
        "Precision": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
        ),
        "Recall": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
        ),
        "F1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
        ),
    }


def evaluate_checkpoint(
    checkpoint_path,
    dataset
):
    model = keras.models.load_model(
        checkpoint_path,
        compile=False
    )

    y_true, probabilities = get_predictions(
        model,
        dataset
    )

    metrics = calculate_metrics(
        y_true,
        probabilities
    )

    return (
        model,
        y_true,
        probabilities,
        metrics
    )


def plot_history(
    history_df,
    title_prefix
):
    epochs = np.arange(
        1,
        len(
            history_df
        ) + 1
    )

    plt.figure(
        figsize=(8, 4)
    )
    plt.plot(
        epochs,
        history_df[
            "accuracy"
        ],
        label="Training Accuracy"
    )
    plt.plot(
        epochs,
        history_df[
            "val_accuracy"
        ],
        label="Validation Accuracy"
    )
    plt.title(
        f"{title_prefix} — Accuracy"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(
        figsize=(8, 4)
    )
    plt.plot(
        epochs,
        history_df[
            "loss"
        ],
        label="Training Loss"
    )
    plt.plot(
        epochs,
        history_df[
            "val_loss"
        ],
        label="Validation Loss"
    )
    plt.title(
        f"{title_prefix} — Loss"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(
        figsize=(8, 4)
    )
    plt.plot(
        epochs,
        history_df[
            "f1_macro"
        ],
        label="Training Macro F1"
    )
    plt.plot(
        epochs,
        history_df[
            "val_f1_macro"
        ],
        label="Validation Macro F1"
    )
    plt.title(
        f"{title_prefix} — Macro F1"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 11. Manual Hyperparameter Tuning Plan — Variant A vs Variant B

### Variant A — Strong Transfer Learning
- Dense head: 256
- Dropout: 0.35
- Head LR: 3e-4
- Fine-tune LR: 1e-5
- Fine-tune upper 40 backbone layers
- Light augmentation

### Variant B — Deep Fine-Tuning
- Dense head: 512 → 128
- Batch Normalization
- L2 regularization
- Dropout: 0.45
- Head LR: 1e-4
- Fine-tune LR: 5e-6
- Fine-tune upper 80 backbone layers
- Moderate augmentation

The selected variant is the one with the highest **Validation Macro F1**.

## 12. Build the Two ImageNet Model Variants

In [ ]:
MODEL_NAME = "EfficientNetB0"

VARIANT_NAMES = {
    "A": "Strong Transfer Learning",
    "B": "Deep Fine-Tuning",
}

VARIANTS = {
    "A": {
        "dense_units": 256,
        "second_dense": 0,
        "dropout": 0.35,
        "head_lr": 3e-4,
        "finetune_lr": 1e-5,
        "finetune_layers": 40,
        "weight_decay": 1e-5,
        "l2": 0.0,
        "stage1_epochs": 15,
        "stage2_epochs": 30,
        "augmentation": "light",
    },
    "B": {
        "dense_units": 512,
        "second_dense": 128,
        "dropout": 0.45,
        "head_lr": 1e-4,
        "finetune_lr": 5e-6,
        "finetune_layers": 80,
        "weight_decay": 2e-5,
        "l2": 1e-4,
        "stage1_epochs": 18,
        "stage2_epochs": 35,
        "augmentation": "moderate",
    },
}


def make_augmentation(
    strength
):
    if strength == "light":
        return keras.Sequential(
            [
                layers.RandomFlip(
                    "horizontal"
                ),
                layers.RandomRotation(
                    0.025
                ),
                layers.RandomTranslation(
                    0.025,
                    0.025
                ),
                layers.RandomZoom(
                    0.035
                ),
                layers.RandomContrast(
                    0.05
                ),
            ],
            name="light_augmentation"
        )

    return keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal"
            ),
            layers.RandomRotation(
                0.04
            ),
            layers.RandomTranslation(
                0.04,
                0.04
            ),
            layers.RandomZoom(
                0.05
            ),
            layers.RandomContrast(
                0.08
            ),
        ],
        name="moderate_augmentation"
    )


def build_model(
    variant_name
):
    cfg = VARIANTS[
        variant_name
    ]

    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3
        ),
        name="mri_image"
    )

    x = make_augmentation(
        cfg[
            "augmentation"
        ]
    )(inputs)

    backbone = tf.keras.applications.EfficientNetB0(
        input_shape=(
            IMG_SIZE,
            IMG_SIZE,
            3
        ),
        include_top=False,
        weights="imagenet",
        pooling="avg"
    )

    backbone.trainable = False

    x = backbone(
        x,
        training=False
    )

    if variant_name == "B":
        x = layers.BatchNormalization(
            name="head_batch_norm"
        )(x)

    x = layers.Dense(
        cfg[
            "dense_units"
        ],
        activation="swish",
        kernel_regularizer=regularizers.l2(
            cfg[
                "l2"
            ]
        ),
        name="head_dense_1"
    )(x)

    x = layers.Dropout(
        cfg[
            "dropout"
        ],
        name="head_dropout_1"
    )(x)

    if cfg[
        "second_dense"
    ] > 0:
        x = layers.Dense(
            cfg[
                "second_dense"
            ],
            activation="swish",
            kernel_regularizer=regularizers.l2(
                cfg[
                    "l2"
                ]
            ),
            name="head_dense_2"
        )(x)

        x = layers.Dropout(
            0.25,
            name="head_dropout_2"
        )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="tumor_class"
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name=(
            f"{MODEL_NAME}_"
            f"Variant_{variant_name}"
        )
    )

    return (
        model,
        backbone
    )


def set_finetune_layers(
    backbone,
    last_n
):
    backbone.trainable = True

    for layer in backbone.layers:
        layer.trainable = False

    for layer in backbone.layers[
        -last_n:
    ]:
        if not isinstance(
            layer,
            layers.BatchNormalization
        ):
            layer.trainable = True

    print(
        "Trainable backbone layers:",
        sum(
            int(
                layer.trainable
            )
            for layer in backbone.layers
        )
    )


def train_variant(
    variant_name
):
    cfg = VARIANTS[
        variant_name
    ]

    print(
        "TRAINING",
        MODEL_NAME,
        "VARIANT",
        variant_name,
        "—",
        VARIANT_NAMES[
            variant_name
        ]
    )

    model, backbone = build_model(
        variant_name
    )

    compile_model(
        model,
        cfg[
            "head_lr"
        ],
        cfg[
            "weight_decay"
        ]
    )

    stage1_path = (
        MODEL_DIR
        / (
            f"{MODEL_NAME}_"
            f"{variant_name}_"
            "stage1_best.keras"
        )
    )

    history1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=cfg[
            "stage1_epochs"
        ],
        callbacks=make_callbacks(
            stage1_path,
            patience=6
        ),
        verbose=1
    )

    _, _, _, stage1_metrics = evaluate_checkpoint(
        stage1_path,
        val_ds
    )

    model = keras.models.load_model(
        stage1_path,
        compile=False
    )

    backbone = None

    for layer in model.layers:
        if isinstance(
            layer,
            keras.Model
        ):
            backbone = layer

    assert backbone is not None

    set_finetune_layers(
        backbone,
        cfg[
            "finetune_layers"
        ]
    )

    compile_model(
        model,
        cfg[
            "finetune_lr"
        ],
        cfg[
            "weight_decay"
        ]
    )

    stage2_path = (
        MODEL_DIR
        / (
            f"{MODEL_NAME}_"
            f"{variant_name}_"
            "stage2_best.keras"
        )
    )

    history2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=cfg[
            "stage2_epochs"
        ],
        callbacks=make_callbacks(
            stage2_path,
            patience=8
        ),
        verbose=1
    )

    _, _, _, stage2_metrics = evaluate_checkpoint(
        stage2_path,
        val_ds
    )

    stage_results = pd.DataFrame(
        [
            {
                "Stage": "Frozen Backbone",
                **stage1_metrics
            },
            {
                "Stage": "Fine-Tuning",
                **stage2_metrics
            },
        ]
    ).sort_values(
        "F1",
        ascending=False
    ).reset_index(
        drop=True
    )

    if stage_results.loc[
        0,
        "Stage"
    ] == "Frozen Backbone":
        best_path = stage1_path
    else:
        best_path = stage2_path

    result = {
        "Variant": variant_name,
        "Variant Name": VARIANT_NAMES[
            variant_name
        ],
        "Best Stage": str(
            stage_results.loc[
                0,
                "Stage"
            ]
        ),
        "Validation Accuracy": float(
            stage_results.loc[
                0,
                "Accuracy"
            ]
        ),
        "Validation Precision": float(
            stage_results.loc[
                0,
                "Precision"
            ]
        ),
        "Validation Recall": float(
            stage_results.loc[
                0,
                "Recall"
            ]
        ),
        "Validation F1": float(
            stage_results.loc[
                0,
                "F1"
            ]
        ),
        "Best Checkpoint": str(
            best_path
        ),
    }

    return {
        "result": result,
        "history_stage1": pd.DataFrame(
            history1.history
        ),
        "history_stage2": pd.DataFrame(
            history2.history
        ),
        "stage_results": stage_results,
        "best_path": best_path,
    }

## 13. Train Variant A Separately — Strong Transfer Learning

In [ ]:
VARIANT_A = train_variant(
    "A"
)

display(
    pd.DataFrame([
        VARIANT_A[
            "result"
        ]
    ])
)

## 14. Variant A Visualization — Accuracy, Loss and Macro F1

In [ ]:
plot_history(
    VARIANT_A[
        "history_stage1"
    ],
    (
        f"{MODEL_NAME} Variant A — "
        "Frozen Backbone"
    )
)

plot_history(
    VARIANT_A[
        "history_stage2"
    ],
    (
        f"{MODEL_NAME} Variant A — "
        "Fine-Tuning"
    )
)

display(
    VARIANT_A[
        "stage_results"
    ]
)

## 15. Train Variant B Separately — Deep Fine-Tuning

In [ ]:
VARIANT_B = train_variant(
    "B"
)

display(
    pd.DataFrame([
        VARIANT_B[
            "result"
        ]
    ])
)

## 16. Variant B Visualization — Accuracy, Loss and Macro F1

In [ ]:
plot_history(
    VARIANT_B[
        "history_stage1"
    ],
    (
        f"{MODEL_NAME} Variant B — "
        "Frozen Backbone"
    )
)

plot_history(
    VARIANT_B[
        "history_stage2"
    ],
    (
        f"{MODEL_NAME} Variant B — "
        "Fine-Tuning"
    )
)

display(
    VARIANT_B[
        "stage_results"
    ]
)

## 17. Compare Variant A vs Variant B and Select the Best

In [ ]:
variant_results = pd.DataFrame([
    VARIANT_A[
        "result"
    ],
    VARIANT_B[
        "result"
    ],
]).sort_values(
    "Validation F1",
    ascending=False
).reset_index(
    drop=True
)

display(
    variant_results
)

for metric in [
    "Validation Accuracy",
    "Validation Precision",
    "Validation Recall",
    "Validation F1"
]:
    plt.figure(
        figsize=(7, 4)
    )

    plt.bar(
        variant_results[
            "Variant Name"
        ],
        variant_results[
            metric
        ]
    )

    plt.title(
        f"{MODEL_NAME} — {metric}"
    )

    plt.ylabel(
        metric
    )

    plt.xticks(
        rotation=12
    )

    plt.tight_layout()
    plt.show()

BEST_VARIANT = str(
    variant_results.loc[
        0,
        "Variant"
    ]
)

BEST_VARIANT_NAME = str(
    variant_results.loc[
        0,
        "Variant Name"
    ]
)

BEST_CHECKPOINT = Path(
    variant_results.loc[
        0,
        "Best Checkpoint"
    ]
)

FINAL_MODEL_PATH = (
    MODEL_DIR
    / (
        f"{MODEL_NAME}_"
        "BEST_BRAIN_TUMOR_MODEL.keras"
    )
)

shutil.copy2(
    BEST_CHECKPOINT,
    FINAL_MODEL_PATH
)

print(
    "Selected:",
    BEST_VARIANT,
    "—",
    BEST_VARIANT_NAME
)

print(
    "Selection metric:",
    "Validation Macro F1"
)

## 18. Official Test Evaluation, Error Analysis and Result Export

The Official Test set is used only **after** Variant A/B selection.

Outputs:
- selected `.keras` model,
- final result CSV,
- variant comparison CSV,
- classification report,
- confusion matrix,
- metadata JSON.

In [ ]:
final_model = keras.models.load_model(
    FINAL_MODEL_PATH,
    compile=False
)

y_test_true, test_probs = get_predictions(
    final_model,
    test_ds
)

test_metrics = calculate_metrics(
    y_test_true,
    test_probs
)

TEST_METRICS = {
    "Test Accuracy": test_metrics[
        "Accuracy"
    ],
    "Test Precision": test_metrics[
        "Precision"
    ],
    "Test Recall": test_metrics[
        "Recall"
    ],
    "Test F1": test_metrics[
        "F1"
    ],
}

display(
    pd.DataFrame([
        TEST_METRICS
    ])
)

test_pred = np.argmax(
    test_probs,
    axis=1
)

report_dict = classification_report(
    y_test_true,
    test_pred,
    target_names=CLASS_NAMES,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(
    report_dict
).transpose()

display(
    report_df
)

cm = confusion_matrix(
    y_test_true,
    test_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"True {x}"
        for x in CLASS_NAMES
    ],
    columns=[
        f"Pred {x}"
        for x in CLASS_NAMES
    ]
)

display(
    cm_df
)

plt.figure(
    figsize=(8, 7)
)

plt.imshow(
    cm
)

plt.title(
    f"{MODEL_NAME} — Official Test Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "True Class"
)

plt.xticks(
    np.arange(
        NUM_CLASSES
    ),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    np.arange(
        NUM_CLASSES
    ),
    CLASS_NAMES
)

for i in range(
    cm.shape[
        0
    ]
):
    for j in range(
        cm.shape[
            1
        ]
    ):
        plt.text(
            j,
            i,
            str(
                cm[
                    i,
                    j
                ]
            ),
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

FINAL_RESULT_CSV = (
    RESULTS_DIR
    / (
        f"{MODEL_NAME.lower()}_"
        "final_result.csv"
    )
)

VARIANT_CSV = (
    RESULTS_DIR
    / (
        f"{MODEL_NAME.lower()}_"
        "variant_comparison.csv"
    )
)

REPORT_CSV = (
    RESULTS_DIR
    / (
        f"{MODEL_NAME.lower()}_"
        "classification_report.csv"
    )
)

CM_CSV = (
    RESULTS_DIR
    / (
        f"{MODEL_NAME.lower()}_"
        "confusion_matrix.csv"
    )
)

final_result = pd.DataFrame([
    {
        "Model Owner": MODEL_OWNER,
        "IT Number": OWNER_IT_NUMBER,
        "Model": MODEL_NAME,
        "Weights": "ImageNet",
        "Selected Variant": BEST_VARIANT,
        "Selected Variant Name": BEST_VARIANT_NAME,
        "Validation F1": float(
            variant_results.loc[
                0,
                "Validation F1"
            ]
        ),
        **TEST_METRICS,
    }
])

final_result.to_csv(
    FINAL_RESULT_CSV,
    index=False
)

variant_results.to_csv(
    VARIANT_CSV,
    index=False
)

report_df.to_csv(
    REPORT_CSV
)

cm_df.to_csv(
    CM_CSV
)

metadata = {
    "model_owner": MODEL_OWNER,
    "owner_it_number": OWNER_IT_NUMBER,
    "owner_member_no": OWNER_MEMBER_NO,
    "model_name": MODEL_NAME,
    "weights": "ImageNet",
    "selected_variant": BEST_VARIANT,
    "selected_variant_name": BEST_VARIANT_NAME,
    "selection_metric": "Validation Macro F1",
    "validation_f1_macro": float(
        variant_results.loc[
            0,
            "Validation F1"
        ]
    ),
    "test_metrics": TEST_METRICS,
    "class_names": CLASS_NAMES,
    "input_size": [
        IMG_SIZE,
        IMG_SIZE
    ],
    "model_input_range": "0-255 float32",
    "dataset": KAGGLE_DATASET,
    "variant_configs": VARIANTS,
}

METADATA_PATH = (
    RESULTS_DIR
    / "best_model_metadata.json"
)

with open(
    METADATA_PATH,
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

display(
    final_result
)

print(
    "Saved final model:",
    FINAL_MODEL_PATH
)

## 19. Download Final EfficientNetB0 Package + Viva Explanation

**Viva answer:**

> “We used ImageNet transfer learning for EfficientNetB0. We trained two different tuned variants. Each variant first trained a frozen pretrained backbone and then fine-tuned the upper layers with a smaller learning rate. We selected the best variant using Validation Macro F1 and used the Official Test set only for final evaluation.”

The final `.keras` model is downloaded in the last cell for later localhost deployment.

In [ ]:
from google.colab import files

PACKAGE_DIR = (
    WORK_DIR
    / (
        f"{MODEL_NAME}_"
        "FINAL_PACKAGE"
    )
)

if PACKAGE_DIR.exists():
    shutil.rmtree(
        PACKAGE_DIR
    )

PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for src in [
    FINAL_MODEL_PATH,
    FINAL_RESULT_CSV,
    VARIANT_CSV,
    REPORT_CSV,
    CM_CSV,
    METADATA_PATH,
]:
    shutil.copy2(
        src,
        PACKAGE_DIR
        / Path(
            src
        ).name
    )

PACKAGE_ZIP = Path(
    shutil.make_archive(
        f"/content/{MODEL_NAME}_FINAL_PACKAGE",
        "zip",
        root_dir=PACKAGE_DIR
    )
)

files.download(
    str(
        FINAL_MODEL_PATH
    )
)

files.download(
    str(
        FINAL_RESULT_CSV
    )
)

files.download(
    str(
        METADATA_PATH
    )
)

files.download(
    str(
        PACKAGE_ZIP
    )
)